# 2. PSM Feature Vectors

## Goals:

In this notebook, we explore the earliest and simplest approach to encoding mass spectra for machine learning: a **PSM feature vector**.

1. [Understand what a PSM is and why we need a numerical representation.](#21-what-is-a-psm)
2. [See how a database search produces a ranked list of candidate peptides.](#22-database-search)
3. [Explore the 20 hand-crafted features Percolator (2007) uses to represent each PSM.](#23-the-percolator-feature-vector)
4. [Compute a real 20-element vector from the VLDALDSIK PSM.](#24-computing-the-features)
5. [Understand the limitations of this encoding and how later notebooks address them.](#25-limitations-and-what-comes-next)

In [19]:
# @title Run this cell to import all necessary packages
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import math
from util import *

## 2.1 What is a PSM?

In Notebook 1, we saw how a mass spectrum is produced by fragmenting a peptWde and recording the masses of the resulting fragment ions. We also saw a **peptide-spectrum match (PSM)**: a candidate assignment of a peptide sequence to an observed spectrum, produced by a database search tool.

We claimed that **scan 5672** was matched to **VLDALDSIK** from bovine carbonic anhydrase. Let's pick up from there and prove this match more empirically.

In [20]:
mzml_path = 'Data/04-17-23_CA_Tryp_HCD_10min-calib.mzML'
peptide   = 'VLDALDSIK'
scan      = 5672

spectrum = get_MS2_object(mzml_path, scan, peptide=peptide)
plot_MS2(spectrum, title=f'Scan {scan}: {peptide}')

## 2.2 Database search

Note that most database search engines doesn't return a single best match peptide from a spectrum, rather a *ranked list* of candidate peptides for every spectrum, along with scores that describe how well each candidate fits the data.

Let's see the first 10 ranked matches from 'Data/tide-search.target.tsv'

<!-- TODO UPDATE CONTENT DIRECTORY -->

In [21]:
# first 10 rows of data from that tsv
pd.read_csv('Data/tide-search.target.tsv', sep='\t').head(10)

,file,scan,charge,retention time,spectrum precursor m/z,spectrum neutral mass,peptide mass,delta_cn,delta_lcn,xcorr score,...,xcorr rank,distinct matches/spectrum,sequence,modifications,unmodified sequence,protein id,flanking aa,target/decoy,original target sequence,decoy index
0,Data/04-17-23_CA_Tryp_HCD_10min-calib.mzML,5126,2,12.7066,487.2606,972.5066,972.5491,0.0,0.0,0.095173,...,1,1,VLDALDSIK,NaN,VLDALDSIK,sp|P00921|CAH2_BOVIN(159),KT,target,VLDALDSIK,-1
1,Data/04-17-23_CA_Tryp_HCD_10min-calib.mzML,5672,2,13.3072,487.2821,972.5497,972.5491,0.0,0.0,2.275679,...,1,1,VLDALDSIK,NaN,VLDALDSIK,sp|P00921|CAH2_BOVIN(159),KT,target,VLDALDSIK,-1
2,Data/04-17-23_CA_Tryp_HCD_10min-calib.mzML,6178,2,13.8462,487.2823,972.5501,972.5491,0.0,0.0,2.146796,...,1,1,VLDALDSIK,NaN,VLDALDSIK,sp|P00921|CAH2_BOVIN(159),KT,target,VLDALDSIK,-1
3,Data/04-17-23_CA_Tryp_HCD_10min-calib.mzML,3696,2,11.1443,487.2882,972.5619,972.5491,0.0,0.0,0.263293,...,1,1,VLDALDSIK,NaN,VLDALDSIK,sp|P00921|CAH2_BOVIN(159),KT,target,VLDALDSIK,-1
4,Data/04-17-23_CA_Tryp_HCD_10min-calib.mzML,4807,2,12.3661,490.2461,978.4777,978.4770,0.0,0.0,2.249104,...,1,1,DGPLTGTYR,NaN,DGPLTGTYR,sp|P00921|CAH2_BOVIN(81),KL,target,DGPLTGTYR,-1
5,Data/04-17-23_CA_Tryp_HCD_10min-calib.mzML,3831,2,11.2851,490.2464,978.4782,978.4770,0.0,0.0,0.507732,...,1,1,DGPLTGTYR,NaN,DGPLTGTYR,sp|P00921|CAH2_BOVIN(81),KL,target,DGPLTGTYR,-1
6,Data/04-17-23_CA_Tryp_HCD_10min-calib.mzML,4342,2,11.8494,490.2467,978.4789,978.4770,0.0,0.0,2.245130,...,1,1,DGPLTGTYR,NaN,DGPLTGTYR,sp|P00921|CAH2_BOVIN(81),KL,target,DGPLTGTYR,-1
7,Data/04-17-23_CA_Tryp_HCD_10min-calib.mzML,2973,2,10.1499,490.2573,978.5001,978.4770,0.0,0.0,0.261353,...,1,1,DGPLTGTYR,NaN,DGPLTGTYR,sp|P00921|CAH2_BOVIN(81),KL,target,DGPLTGTYR,-1
8,Data/04-17-23_CA_Tryp_HCD_10min-calib.mzML,5889,2,13.5414,501.7399,1001.4652,1001.5029,0.0,0.0,0.194668,...,1,1,QSPVDIDTK,NaN,QSPVDIDTK,sp|P00921|CAH2_BOVIN(28),RA,target,QSPVDIDTK,-1
9,Data/04-17-23_CA_Tryp_HCD_10min-calib.mzML,4068,2,11.5380,501.7592,1001.5039,1001.5029,0.0,0.0,1.927699,...,1,1,QSPVDIDTK,NaN,QSPVDIDTK,sp|P00921|CAH2_BOVIN(28),RA,target,QSPVDIDTK,-1


We assumed beforehand that scan 5672 was the peptide "VLDALDSIK," but why is that?

Looking at the table above, no single score cleanly divides correct from incorrect PSMs. A high XCorr doesn't guarantee a correct match; a low one doesn't rule it out.

[Percolator (Käll et al., 2007)](https://www.nature.com/articles/nmeth1113) addressed this by collecting multiple scores into a fixed-length vector, then training a [support vector machine (SVM)](https://en.wikipedia.org/wiki/Support_vector_machine) on target and decoy PSMs to find the optimal decision boundary.

## 2.2.1 Why a numerical representation?

Machine learning models are mathematical functions. They cannot accept a raw spectrum, a peptide string like `VLDALDSIK`, or a search-engine report as input. They require a **fixed-length vector of numbers**.

This is the central, simplest form of an encoding problem: how do you convert a PSM which contains a spectrum, a candidate peptide, and a set of search scores into a single numerical vector that a model can work with?

Percolator's answer was pragmatic: hand-pick quantities you already trust (XCorr, mass error, ion fraction, charge, etc.), compute them for every PSM, and stack the results into a 20-number vector. Every PSM, correct or incorrect, gets compressed to the same 20 dimensions, and the SVM learns which combinations of those numbers distinguish good matches from bad ones.

## 2.3 The Percolator Feature Vector

Here are the aforementioned features, taken directly from the Percolator paper:

| # | Feature | Description |
|---|---------|-------------|
| 1 | XCorr | Cross correlation between calculated and observed spectra |
| 2 | $\Delta C_n$ | Fractional difference between current and second best XCorr |
| 3 | $\Delta C_n^L$ | Fractional difference between current and fifth best XCorr |
| 4 | Sp | Preliminary score for peptide versus predicted fragment ion values |
| 5 | $\ln(\text{rSp})$ | The natural logarithm of the rank of the match based on the Sp score |
| 8 | Mass | The observed mass $[\text{M+H}]^+$ |
| 6 | $\Delta M$ | The difference in calculated and observed mass |
| 7 | $\text{abs}(\Delta M)$ | The absolute value of the difference in calculated and observed mass |
| 9 | ionFrac | The fraction of matched b and y ions |
| 10 | $\ln(\text{NumSp})$ | The natural logarithm of the number of database peptides within the specified m/z range |
| 11 | enzN | Boolean: Is the peptide preceded by an enzymatic (tryptic) site? |
| 12 | enzC | Boolean: Does the peptide have an enzymatic (tryptic) C-terminus? |
| 13 | enzInt | Number of missed internal enzymatic (tryptic) sites |
| 14 | pepLen | The length of the matched peptide, in residues |
| 15–17 | charge1–3 | Three Boolean features indicating the charge state |
| 18 | $\ln(\text{numPep})$ | Number of PSMs for which this is the best scoring peptide. |
| 19 | $\ln(\text{numProt})$ | Number of times the matched protein matches other PSMs. |
| 20 | $\ln(\text{pepSite})$ | Number of different peptides that match this protein. |


We'll explore this data some more and create this 20 dimensional vector, but try not to get too bogged down in the technical details of how our search engine came up with these magic numbers. 

## 2.4 Computing the Features

Let's build the 20-element Percolator vector for our running example: **scan 5672 → VLDALDSIK**.

We'll work in groups: scoring features, mass features, ion-matching features, enzymatic features, charge features, and protein-level features. At the end, we'll concatenate everything into a single list of 20 numbers.

First, let's load the search results and isolate the row for our PSM.

In [22]:
search = pd.read_csv('Data/tide-search.target.tsv', sep='\t')

psm = search.query('scan == @scan and sequence == @peptide').iloc[0]
psm

file                         Data/04-17-23_CA_Tryp_HCD_10min-calib.mzML
scan                                                               5672
charge                                                                2
retention time                                                  13.3072
spectrum precursor m/z                                         487.2821
spectrum neutral mass                                          972.5497
peptide mass                                                   972.5491
delta_cn                                                            0.0
delta_lcn                                                           0.0
xcorr score                                                    2.275679
tailor score                                                   1.294216
b/y ions matched                                                     10
b/y ions total                                                       16
b/y ions fraction                                               

### Features 1 to 5: Scoring features

These five features come directly out of the database search. They summarize *how well* the candidate peptide explains the spectrum, relative to other candidates the search engine considered.

| # | Feature | Source column |
|---|---------|---------------|
| 1 | XCorr | `xcorr score` |
| 2 | $\Delta C_n$ | `delta_cn` |
| 3 | $\Delta C_n^L$ | `delta_lcn` |
| 4 | Sp | `tailor score` *(Tide's Sp-equivalent)* |
| 5 | $\ln(\text{rSp})$ | $\ln$ of `xcorr rank` |


> *Note: Percolator was originally written for SEQUEST, which produces a separate preliminary score "Sp". Tide doesn't report Sp directly, so we use the **tailor score** as a stand-in, and the xcorr rank in place of rSp.*

In [23]:
xcorr     = psm['xcorr score']
delta_cn  = psm['delta_cn']
delta_lcn = psm['delta_lcn']
sp        = psm['tailor score']
ln_rSp    = math.log(psm['xcorr rank'])

print(f'XCorr      = {xcorr:.4f}')
print(f'ΔCn        = {delta_cn:.4f}')
print(f'ΔCn^L      = {delta_lcn:.4f}')
print(f'Sp (proxy) = {sp:.4f}')
print(f'ln(rSp)    = {ln_rSp:.4f}')

XCorr      = 2.2757
ΔCn        = 0.0000
ΔCn^L      = 0.0000
Sp (proxy) = 1.2942
ln(rSp)    = 0.0000


### Features 6 to 8: Mass features

Every peptide has a theoretical mass we can compute from its amino-acid sequence. The mass spectrometer also gives us an *observed* mass from the precursor m/z. A correct PSM should match closely; a wild mismatch is a red flag.

| # | Feature | Meaning |
|---|---------|---------|
| 6 | $\Delta M$ | (calculated mass) − (observed mass) |
| 7 | $\lvert\Delta M\rvert$ | absolute mass error |
| 8 | Mass | observed $[M+H]^+$ |

The TSV gives us the neutral mass; we add one proton to get $[M+H]^+$.

In [24]:
PROTON_MASS = 1.00727646677
# in Daltons (Da)
observed_mass = psm['spectrum neutral mass']
calc_mass     = psm['peptide mass']

delta_M     = calc_mass - observed_mass
abs_delta_M = abs(delta_M)
mass_MH     = observed_mass + PROTON_MASS   # [M+H]+

print(f'ΔM         = {delta_M:+.4f} Da')
print(f'|ΔM|       = {abs_delta_M:.4f} Da')
print(f'[M+H]+     = {mass_MH:.4f} Da')

ΔM         = -0.0006 Da
|ΔM|       = 0.0006 Da
[M+H]+     = 973.5570 Da


### Features 9 to 10: Ion-matching and search-space features

| # | Feature | Meaning |
|---|---------|---------|
| 9 | ionFrac | fraction of theoretical b/y ions actually observed |
| 10 | $\ln(\text{NumSp})$ | $\ln$ of the number of database peptides within the precursor m/z window |

Tide gives us `b/y ions fraction` directly. For NumSp we'll use `distinct matches/spectrum`: the number of distinct peptides Tide scored against this spectrum.

In [25]:
ion_frac  = psm['b/y ions fraction']
ln_NumSp  = math.log(psm['distinct matches/spectrum'])

print(f'ionFrac    = {ion_frac:.4f}  ({psm["b/y ions matched"]}/{psm["b/y ions total"]} ions)')
print(f'ln(NumSp)  = {ln_NumSp:.4f}')

ionFrac    = 0.6250  (10/16 ions)
ln(NumSp)  = 0.0000


### Features 11 to 14: Enzymatic and length features

Trypsin (the enzyme used to digest our protein) cleaves *after* lysine (K) or arginine (R), unless followed by proline (P). A "well-behaved" tryptic peptide therefore has:

- a K or R immediately *before* it in the parent protein (enzymatic N-terminus), and
- a K or R at its *C-terminus* (enzymatic C-terminus),
- and no internal K/R (no missed cleavages).

| # | Feature | Meaning |
|---|---------|---------|
| 11 | enzN | preceded by a tryptic site? |
| 12 | enzC | ends in K or R? |
| 13 | enzInt | number of internal missed cleavages |
| 14 | pepLen | peptide length |

The `flanking aa` column gives us the two residues that flank the peptide in the parent protein: the residue *before* it and the residue *after* it. For VLDALDSIK the flanking residues are `KT`, meaning the protein context is `...K-VLDALDSIK-T...`.

In [26]:
flank_before, flank_after = psm['flanking aa'][0], psm['flanking aa'][1]
seq = psm['unmodified sequence']

# Tryptic N-term: previous residue is K or R (or peptide starts the protein, '-')
enzN = int(flank_before in ('K', 'R', '-'))

# Tryptic C-term: peptide ends in K or R (or is the protein's C-terminus)
enzC = int(seq[-1] in ('K', 'R') or flank_after == '-')

# Missed internal cleavages: K or R inside the peptide (not at the C-terminus),
# and not followed by P
enzInt = sum(
    1 for i, aa in enumerate(seq[:-1])
    if aa in ('K', 'R') and seq[i + 1] != 'P'
)

pepLen = len(seq)

print(f'flanking   = {flank_before}-[{seq}]-{flank_after}')
print(f'enzN       = {enzN}')
print(f'enzC       = {enzC}')
print(f'enzInt     = {enzInt}')
print(f'pepLen     = {pepLen}')

flanking   = K-[VLDALDSIK]-T
enzN       = 1
enzC       = 1
enzInt     = 0
pepLen     = 9


### Features 15 to 17: Charge state (one-hot)

Percolator represents the precursor charge as three boolean features. Our precursor is **2+**, so we expect `[0, 1, 0]`.

In [27]:
z = int(psm['charge'])

charge1 = int(z == 1)
charge2 = int(z == 2)
charge3 = int(z == 3)

print(f'charge     = {z}+  ->  [{charge1}, {charge2}, {charge3}]')

charge     = 2+  ->  [0, 1, 0]


### Features 18 to 20: Protein- and peptide-level counts

The final three features look beyond a single PSM and ask: how *popular* is this peptide and its parent protein across the whole search?

| # | Feature | Meaning |
|---|---------|---------|
| 18 | $\ln(\text{numPep})$ | how many spectra picked this peptide as their top match |
| 19 | $\ln(\text{numProt})$ | how many PSMs match this peptide's parent protein |
| 20 | $\ln(\text{pepSite})$ | how many *distinct* peptides match this protein |

A real, abundant peptide tends to show up across many spectra, such that high values here are evidence that the identification is genuine.

We compute these by aggregating over the entire search table.

In [28]:
protein = psm['protein id']

# numPep: PSMs (rank-1 matches) where the chosen peptide is this peptide
numPep  = (search['sequence'] == peptide).sum()

# numProt: PSMs whose top match falls on this protein
numProt = (search['protein id'] == protein).sum()

# pepSite: distinct peptides assigned to this protein
pepSite = search.loc[search['protein id'] == protein, 'sequence'].nunique()

ln_numPep  = math.log(numPep)
ln_numProt = math.log(numProt)
ln_pepSite = math.log(pepSite)

print(f'numPep     = {numPep}   ->  ln = {ln_numPep:.4f}')
print(f'numProt    = {numProt}   ->  ln = {ln_numProt:.4f}')
print(f'pepSite    = {pepSite}    ->  ln = {ln_pepSite:.4f}')

numPep     = 4   ->  ln = 1.3863
numProt    = 4   ->  ln = 1.3863
pepSite    = 1    ->  ln = 0.0000


### Putting it all together

We now have all 20 numbers. Concatenated in the order from the Percolator paper, this single Python list **is** our PSM feature vector, the same shape of vector Percolator would feed to its machine learning models to consider *all* of this information when ranking peptides.

That is, Percolator runs a first pass database search, then reranks the matches based computations given the first pass.

In [29]:
feature_vector = [
    xcorr,         # 1.  XCorr
    delta_cn,      # 2.  ΔCn
    delta_lcn,     # 3.  ΔCn^L
    sp,            # 4.  Sp
    ln_rSp,        # 5.  ln(rSp)
    delta_M,       # 6.  ΔM
    abs_delta_M,   # 7.  |ΔM|
    mass_MH,       # 8.  Mass [M+H]+
    ion_frac,      # 9.  ionFrac
    ln_NumSp,      # 10. ln(NumSp)
    enzN,          # 11. enzN
    enzC,          # 12. enzC
    enzInt,        # 13. enzInt
    pepLen,        # 14. pepLen
    charge1,       # 15. charge1
    charge2,       # 16. charge2
    charge3,       # 17. charge3
    ln_numPep,     # 18. ln(numPep)
    ln_numProt,    # 19. ln(numProt)
    ln_pepSite,    # 20. ln(pepSite)
]

assert len(feature_vector) == 20
feature_vector

[np.float64(2.27567854),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(1.2942159),
 0.0,
 np.float64(-0.0006000000000767614),
 np.float64(0.0006000000000767614),
 np.float64(973.55697646677),
 np.float64(0.625),
 0.0,
 1,
 1,
 0,
 9,
 0,
 1,
 0,
 1.3862943611198906,
 1.3862943611198906,
 0.0]

## 2.5 Limitations and What Comes Next

The 20-number vector works and Percolator significantly improves identification rates over raw search scores alone, but it comes at a price.

That is to say, these features are **hand-crafted**: they encode exactly the information that experts in 2007 judged to be informative about a PSM. 

I want to get you thinking about signals that didn't make the list.

Peak shape, isotope envelope details, the spatial arrangement of fragment ions, are all discarded. The encoding is also **lossy and order-agnostic**: two completely different spectra that happen to share the same XCorr, mass error, and ion fraction collapse to exactly the same point in vector space.

The remaining notebooks explore what happens when you replace hand-crafted numbers with representations learned from the raw spectrum itself. 

Each further tutorial trades these expert assumptions for learned representations, at increasing computational cost. This PSM feature vector is the baseline. It's quite fast, interpretable, and still the most widely used, but fundamentally limited by what 20 numbers can see.